# Nguyễn Hoàng Trọng Sơn - 22521252

# Import library

In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, year, month, desc, count, avg, when, countDistinct, max, min, datediff, rank

In [2]:
spark = SparkSession.builder \
    .appName("Fecom Analytics") \
    .getOrCreate()

# 1.	Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột.

In [3]:
customer_df = spark.read.csv("/content/Customer_List.csv",  sep=";", header=True)
orders_df   = spark.read.csv("/content/Orders.csv",         sep=";", header=True)
items_df    = spark.read.csv("/content/Order_Items.csv",    sep=";", header=True)
products_df = spark.read.csv("/content/Products.csv",       sep=";", header=True)
reviews_df  = spark.read.csv("/content/Order_Reviews.csv",  sep=";", header=True)

In [4]:
customer_df.show(10)

+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|     Customer_Trx_ID|       Subscriber_ID|Subscribe_Date|First_Order_Date|Customer_Postal_Code|Customer_City|Customer_Country|Customer_Country_Code|Age|Gender|
+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|1e959e1f5920cba43...|9765e039028279fd2...|    2023-07-08|      2023-07-09|            FR-75005|        Paris|          France|                   FR| 29|  Male|
|9877437582f263da7...|a75e134e7eb6f96e2...|    2024-03-23|      2024-04-11|           PL-00-001|       Warsaw|          Poland|                   PL| 38|  Male|
|fa6fbbb2080646aca...|2fdac27295500e820...|    2023-05-12|      2023-06-01|             NL-1012|    Amsterdam|     Netherlands|                   NL| 35|Female|
|a4c9ff14ae7620126...|e9ab8fd8ea96

In [5]:
orders_df.show(10)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|            Order_ID|     Customer_Trx_ID|Order_Status|Order_Purchase_Timestamp|Order_Approved_At|Order_Delivered_Carrier_Date|Order_Delivered_Customer_Date|Order_Estimated_Delivery_Date|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|        2023-10-02 10:56| 2023-10-02 11:07|            2023-10-04 19:55|             2023-10-10 21:25|             2023-10-18 00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|        2024-07-24 20:41| 2024-07-26 03:24|            2024-07-26 14:31|             2024-08-07 15:27|             2024-08-13 00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered

In [6]:
items_df.show(10)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            Order_ID|Order_Item_ID|          Product_ID|           Seller_ID|Shipping_Limit_Date| Price|Freight_Value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|   2023-09-19 09:45| 58.90|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|   2023-05-03 11:05|239.90|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|   2024-01-18 14:48|199.00|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|   2024-08-15 10:10| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|   2023-02-13 13:57|199.90|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [7]:
products_df.show(10)

+--------------------+---------------------+-----------------+-----------------+-----------------+----------------+
|          Product_ID|Product_Category_Name|Product_Weight_Gr|Product_Length_Cm|Product_Height_Cm|Product_Width_Cm|
+--------------------+---------------------+-----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|            Perfumery|              225|               16|               10|              14|
|3aa071139cb16b67c...|                  Art|             1000|               30|               18|              20|
|96bd76ec8810374ed...|       Sports_Leisure|              154|               18|                9|              15|
|cef67bcfe19066a93...|                 Baby|              371|               26|                4|              26|
|9dc1a7de274444849...|           Housewares|              625|               20|               17|              13|
|41d3672d4792049fa...|  Musical_Instruments|              200|          

In [8]:
reviews_df.show(10)

+--------------------+--------------------+------------+-----------------------+-------------------------+--------------------+-----------------------+
|           Review_ID|            Order_ID|Review_Score|Review_Comment_Title_En|Review_Comment_Message_En|Review_Creation_Date|Review_Answer_Timestamp|
+--------------------+--------------------+------------+-----------------------+-------------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                   NULL|                     NULL|    2024-01-18 00:00|       2024-01-18 21:46|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                   NULL|                     NULL|    2024-03-10 00:00|       2024-03-11 03:05|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                   NULL|                     NULL|    2024-02-17 00:00|       2024-02-18 14:36|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                   NULL|     I r

# 2.	Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.

In [9]:
total_orders = orders_df.count()
distinct_customers = orders_df.select("Customer_Trx_ID").distinct().count()
distinct_sellers = items_df.select("Seller_ID").distinct().count()

print(f"Tổng đơn hàng: {total_orders}")
print(f"Số khách hàng: {distinct_customers}")
print(f"Số người bán: {distinct_sellers}")

Tổng đơn hàng: 99441
Số khách hàng: 99441
Số người bán: 3095


# 3.	Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần.

In [10]:
orders_with_country = orders_df.join(customer_df, "Customer_Trx_ID", "left")

orders_by_country = (
    orders_with_country
      .groupBy("Customer_Country")
      .agg(count("*").alias("num_orders"))
      .orderBy(desc("num_orders"))
)

orders_by_country.show(50, truncate=False)

+----------------+----------+
|Customer_Country|num_orders|
+----------------+----------+
|Germany         |41754     |
|France          |12848     |
|Netherlands     |11629     |
|Belgium         |5464      |
|Austria         |5043      |
|Switzerland     |3640      |
|United Kingdom  |3382      |
|Poland          |2139      |
|Czechia         |2034      |
|Italy           |2025      |
|Spain           |1651      |
|Portugal        |1336      |
|Sweden          |975       |
|Denmark         |905       |
|Serbia          |746       |
|Norway          |716       |
|Slovakia        |534       |
|Slovenia        |495       |
|Turkey          |485       |
|Greece          |412       |
|Lithuania       |351       |
|Latvia          |280       |
|Croatia         |254       |
|Estonia         |148       |
|Finland         |81        |
|Luxembourg      |68        |
|Andorra         |46        |
+----------------+----------+



# 4.	Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)

In [11]:
orders_time = (
    orders_df
      .withColumn("year",  year(col("Order_Purchase_Timestamp")))
      .withColumn("month", month(col("Order_Purchase_Timestamp")))
      .groupBy("year", "month")
      .agg(count("*").alias("num_orders"))
      .orderBy(col("year").asc(), col("month").desc())
)

orders_time.show(100)

+----+-----+----------+
|year|month|num_orders|
+----+-----+----------+
|2022|   12|         1|
|2022|   10|       324|
|2022|    9|         4|
|2023|   12|      5673|
|2023|   11|      7544|
|2023|   10|      4631|
|2023|    9|      4285|
|2023|    8|      4331|
|2023|    7|      4026|
|2023|    6|      3245|
|2023|    5|      3700|
|2023|    4|      2404|
|2023|    3|      2682|
|2023|    2|      1780|
|2023|    1|       800|
|2024|   10|         4|
|2024|    9|        16|
|2024|    8|      6512|
|2024|    7|      6292|
|2024|    6|      6167|
|2024|    5|      6873|
|2024|    4|      6939|
|2024|    3|      7211|
|2024|    2|      6728|
|2024|    1|      7269|
+----+-----+----------+



# 5.	Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5).
## Lưu ý: Cần xử lý các giá trị ngoại lệ và NULL trong cột Review_Score


In [12]:
clean_reviews = (
    reviews_df
      .filter(col("Review_Score").between(1, 5))
      .na.drop(subset=["Review_Score"])
)

avg_score = clean_reviews.agg(avg("Review_Score").alias("avg_score")).first()["avg_score"]
print(f"Điểm đánh giá trung bình chung: {avg_score:.2f}")

review_counts = (
    clean_reviews
      .groupBy("Review_Score")
      .agg(count("*").alias("count_reviews"))
      .orderBy("Review_Score")
)

review_counts.show()

Điểm đánh giá trung bình chung: 4.09
+------------+-------------+
|Review_Score|count_reviews|
+------------+-------------+
|           1|        11424|
|           2|         3151|
|           3|         8179|
|           4|        19141|
|           5|        57328|
+------------+-------------+



# 6.	Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm

In [13]:
order_items_full = (
    items_df.alias("i")
    .join(orders_df.select("Order_ID", "Order_Purchase_Timestamp"), "Order_ID", "inner")
    .join(products_df.select("Product_ID", "Product_Category_Name"), "Product_ID", "left")
)

rev_2024 = (
    order_items_full
    .withColumn("year", year(col("Order_Purchase_Timestamp")))
    .filter(col("year") == 2024)
    .withColumn("revenue", col("Price") + col("Freight_Value"))
)

revenue_by_category = (
    rev_2024
    .groupBy("Product_Category_Name")
    .sum("revenue")
    .withColumnRenamed("sum(revenue)", "total_revenue")
    .orderBy(desc("total_revenue"))
)

revenue_by_category.show(truncate=False)

+-------------------------------+------------------+
|Product_Category_Name          |total_revenue     |
+-------------------------------+------------------+
|Health_Beauty                  |885191.119999997  |
|Watches_Gifts                  |771986.750000001  |
|Bed_Bath_Table                 |650794.700000002  |
|Sports_Leisure                 |621999.3399999994 |
|Computers_Accessories          |594771.0400000002 |
|Housewares                     |491576.9600000012 |
|Furniture_Decor                |476466.1300000007 |
|Auto                           |404210.5700000002 |
|Baby                           |299052.5599999998 |
|Cool_Stuff                     |273910.0500000001 |
|Garden_Tools                   |259068.31999999983|
|Telephony                      |217452.1299999995 |
|Perfumery                      |204562.53999999992|
|Toys                           |200634.07000000007|
|Office_Furniture               |181745.7300000001 |
|Stationery                     |164743.849999

# 7.	Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm

In [14]:
product_sales = (
    items_df
      .groupBy("Product_ID")
      .count()
      .withColumnRenamed("count", "units_sold")
)

prod_info = products_df.select("Product_ID", "Product_Category_Name")
product_sales = product_sales.join(prod_info, on="Product_ID", how="left")
top_product = product_sales.orderBy(desc("units_sold")).limit(1)
top_product.show()

reviews_clean = (
    reviews_df
      .filter(col("Review_Score").between(1, 5))
      .na.drop(subset=["Review_Score"])
)

prod_reviews = (
    items_df.select("Order_ID", "Product_ID")
      .join(
          reviews_clean.select("Order_ID", "Review_Score"),
          on="Order_ID", how="inner"
      )
      .groupBy("Product_ID")
      .agg(avg("Review_Score").alias("avg_review_score"))
      .join(prod_info, on="Product_ID", how="left")
)

sales_and_reviews = (
    product_sales
      .join(
          prod_reviews.select("Product_ID", "avg_review_score"),
          on="Product_ID", how="left"
      )
      .select("Product_ID", "Product_Category_Name", "units_sold", "avg_review_score")
      .orderBy(desc("units_sold"))
)

sales_and_reviews.show(10, truncate=False)

+--------------------+----------+---------------------+
|          Product_ID|units_sold|Product_Category_Name|
+--------------------+----------+---------------------+
|aca2eb7d00ea1a7b8...|       527|      Furniture_Decor|
+--------------------+----------+---------------------+

+--------------------------------+---------------------+----------+------------------+
|Product_ID                      |Product_Category_Name|units_sold|avg_review_score  |
+--------------------------------+---------------------+----------+------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|Furniture_Decor      |527       |4.019083969465649 |
|99a4788cb24856965c36a24e339b6058|Bed_Bath_Table       |488       |3.8983402489626555|
|422879e10f46682990de24d770e7f83d|Garden_Tools         |484       |3.9465020576131686|
|389d119b48cf3043d311335e499d9c6b|Garden_Tools         |392       |4.117647058823529 |
|368c6c730842d78016ad823897a372db|Garden_Tools         |388       |3.922680412371134 |
|53759a2ecddad2bb87a079

# 8.	Tính toán hiệu số giữa ngày giao hàng thực tế (Order_Delivered_Carrier_Date) và ngày giao hàng dự kiến (ví dụ: Shipping_Limit_Date từ bảng Order_Items) để đánh giá hiệu suất giao hàng.

In [15]:
delivery_perf = (
    items_df.select("Order_ID", "Shipping_Limit_Date")
      .join(
          orders_df.select("Order_ID", "Order_Delivered_Carrier_Date"),
          "Order_ID",
          "inner"
      )
      .withColumn(
          "delay_days",
          datediff(
              col("Order_Delivered_Carrier_Date"),
              col("Shipping_Limit_Date")
          )
      )
)

delivery_perf.select(
    "Order_ID",
    "Shipping_Limit_Date",
    "Order_Delivered_Carrier_Date",
    "delay_days"
).show(10, truncate=False)

+--------------------------------+-------------------+----------------------------+----------+
|Order_ID                        |Shipping_Limit_Date|Order_Delivered_Carrier_Date|delay_days|
+--------------------------------+-------------------+----------------------------+----------+
|00010242fe8c5a6d1ba2dd792cb16214|2023-09-19 09:45   |2023-09-19 18:34            |0         |
|00018f77f2f0320c557190d7a144bdd3|2023-05-03 11:05   |2023-05-04 14:35            |1         |
|000229ec398224ef6ca0657da4fc703e|2024-01-18 14:48   |2024-01-16 12:36            |-2        |
|00024acbcdf0a6daa1e931b038114c75|2024-08-15 10:10   |2024-08-10 13:28            |-5        |
|00042b26cf59d7ce69dfabb4e55b4fd9|2023-02-13 13:57   |2023-02-16 09:46            |3         |
|00048cc3ae777c65dbb7d2a0634bc1ea|2023-05-23 03:55   |2023-05-17 11:05            |-6        |
|00054e8431b9d7675808bcb819fb4a32|2023-12-14 12:10   |2023-12-12 01:07            |-2        |
|000576fe39319847cbb9d288c5617fa6|2024-07-10 12:30

# 9.	Nhóm khách hàng dựa trên số lượng đơn hàng, giá trị trung bình của đơn hàng và tần suất mua sắm.

In [16]:
order_revenue = (
    items_df
      .withColumn("revenue", col("Price") + col("Freight_Value"))
      .groupBy("Order_ID")
      .sum("revenue")
      .withColumnRenamed("sum(revenue)", "order_revenue")
)

cust_orders = (
    orders_df.select("Order_ID", "Customer_Trx_ID", "Order_Purchase_Timestamp")
      .join(order_revenue, "Order_ID", "inner")
)

cust_raw = (
    cust_orders
      .groupBy("Customer_Trx_ID")
      .agg(
          countDistinct("Order_ID").alias("num_orders"),
          avg("order_revenue").alias("avg_order_value"),
          max("Order_Purchase_Timestamp").alias("last_order"),
          min("Order_Purchase_Timestamp").alias("first_order")
      )
)

cust_metrics = cust_raw.withColumn(
    "frequency",
    col("num_orders") / (datediff(col("last_order"), col("first_order")) + 1)
)

cust_metrics.show(10, truncate=False)

+--------------------------------+----------+-----------------+----------------+----------------+---------+
|Customer_Trx_ID                 |num_orders|avg_order_value  |last_order      |first_order     |frequency|
+--------------------------------+----------+-----------------+----------------+----------------+---------+
|000161a058600d5901f007fab4c27140|1         |67.41            |2023-07-16 09:40|2023-07-16 09:40|1.0      |
|00050bf6e01e69d5c0fd612f1bcfb69c|1         |85.22999999999999|2023-09-17 16:04|2023-09-17 16:04|1.0      |
|000598caf2ef4117407665ac33275130|1         |1255.71          |2024-08-11 12:14|2024-08-11 12:14|1.0      |
|0005aefbb696d34b3424dccd0a0e9fd0|1         |147.33           |2024-06-20 09:46|2024-06-20 09:46|1.0      |
|0009a69b72033b2d0ec8c69fc70ef768|1         |173.6            |2023-04-28 13:36|2023-04-28 13:36|1.0      |
|000bf8121c3412d3057d32371c5d3395|1         |45.56            |2023-10-11 07:44|2023-10-11 07:44|1.0      |
|000e943451fc2788ca6ac98a682

# 10.	Xếp hạng các seller dựa trên tổng doanh thu và số lượng đơn hàng bán được.

In [17]:
seller_revenue = (
    items_df
      .withColumn("revenue", col("Price") + col("Freight_Value"))
      .groupBy("Seller_ID")
      .sum("revenue")
      .withColumnRenamed("sum(revenue)", "total_revenue")
)

seller_orders = (
    items_df
      .groupBy("Seller_ID")
      .count()
      .withColumnRenamed("count", "total_orders")
)

seller_metrics = seller_revenue.join(seller_orders, "Seller_ID")

w = Window.orderBy(desc("total_revenue"))
seller_ranked = (
    seller_metrics
      .withColumn("revenue_rank", rank().over(w))
      .orderBy(desc("total_revenue"))
)

seller_ranked.show(10, truncate=False)

+--------------------------------+------------------+------------+------------+
|Seller_ID                       |total_revenue     |total_orders|revenue_rank|
+--------------------------------+------------------+------------+------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|249640.69999999992|1156        |1           |
|7c67e1448b00f6e969d365cea6b010ab|239536.44000000006|1364        |2           |
|53243585a1d6dc2643021fd1853d8905|235856.6800000001 |410         |3           |
|4a3ca9315b744ce9f8e9374361493884|235539.9599999999 |1987        |4           |
|fa1c13f2614d7b5c4749cbc52fecda94|204084.7300000001 |586         |5           |
|da8622b14eb17ae2831f4ac5b9dab84a|185192.32000000018|1551        |6           |
|7e93a43ef30c4f03f38b393420bc753a|182754.05000000005|340         |7           |
|1025f0e2d44d7041d6cf58b6550e0bfa|172860.69         |1428        |8           |
|7a67c85e85bb2ce8582c35f2203ad736|162648.3799999999 |1171        |9           |
|955fee9216a65b617aa5c0531780ce60|160602